# Query → Qwen3 Embedding 0.6B · 1024 · FP16

**Runtime → Change runtime type → T4 GPU → Save**, rồi **Runtime → Run all**.
Tải lên `query_rewritten_weighted.parquet` khi được yêu cầu.

- Giữ nguyên tên/thứ tự tất cả các cột và thứ tự 1.200 dòng đầu vào.
- Chỉ thay `query`, `english`, `chinese`: text → vector 1024 chiều **float16**.
- `id`, `dense_weight` và các cột khác giữ nguyên giá trị và kiểu dữ liệu.
- Model chạy FP16; last-token pooling với left padding; chuẩn hóa L2 ở FP32
  rồi chuyển vector sang FP16 trước khi lưu Parquet.
- Thêm instruction truy xuất ở đầu mỗi query theo hướng dẫn Qwen.
  Instruction chỉ dùng khi inference, không tạo thêm cột hay nhân `dense_weight` vào vector.
- Xử lý theo batch, không cắt text; lỗi nếu vượt giới hạn token. OOM được chia batch.
- Sau khi kiểm tra output, bấm **Tải file Parquet FP16** để tải `query_rewritten_weighted_qwen3_1024_fp16.parquet`.

Model/revision khớp các notebook ViBioMIR Qwen3 hiện có trong dự án.
Tài liệu chính thức: [Qwen model card](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B).


In [ ]:
# Giữ PyTorch/CUDA có sẵn của Colab; không cài FlashAttention trên T4.
import subprocess
import sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'pyarrow>=21.0.0', 'tqdm>=4.66',
])


In [ ]:
from pathlib import Path
from itertools import zip_longest
from time import perf_counter
import gc
import hashlib
import json
import tempfile

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

INPUT_PATH = '/content/query_rewritten_weighted.parquet'
OUTPUT_PATH = ''  # Để trống: lưu cạnh input, thêm hậu tố _qwen3_1024_fp16.
TEXT_COLUMNS = ('query', 'english', 'chinese')
ROW_BATCH_SIZE = 128
ENCODER_BATCH_SIZE = 32
PADDED_TOKEN_BUDGET = 8192
DOWNLOAD_OUTPUT = True

ENCODER = {
    'model': 'Qwen/Qwen3-Embedding-0.6B',
    'revision': '97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3',
    'dimension': 1024,
    'max_length': 4096,
    'inference_dtype': 'float16',
    'storage_dtype': 'float16',
    'pooling': 'last_token',
    'padding_side': 'left',
    'normalized': True,
    'query_instruction': 'Given a web search query, retrieve relevant passages that answer the query',
}


In [ ]:
if not INPUT_PATH or not Path(INPUT_PATH).is_file():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Chọn đúng một file Parquet.')
    INPUT_PATH = next(iter(uploaded))
    del uploaded

input_path = Path(INPUT_PATH).resolve()
output_path = (Path(OUTPUT_PATH).resolve() if OUTPUT_PATH else
               input_path.with_name(input_path.stem + '_qwen3_1024_fp16.parquet'))
if input_path == output_path:
    raise ValueError('Output phải khác input để giữ file text gốc.')
parquet = pq.ParquetFile(input_path)
if not parquet.metadata.num_rows:
    raise ValueError('Input không có dòng dữ liệu.')
if len(set(parquet.schema_arrow.names)) != len(parquet.schema_arrow.names):
    raise ValueError('Tên cột bị trùng.')
for name in TEXT_COLUMNS:
    field = parquet.schema_arrow.field(name)
    if not (pa.types.is_string(field.type) or pa.types.is_large_string(field.type)):
        raise ValueError(f'{name} phải là cột text.')
for batch in parquet.iter_batches(batch_size=ROW_BATCH_SIZE, columns=list(TEXT_COLUMNS)):
    for name in TEXT_COLUMNS:
        if any(value is None or not value.strip() for value in batch.column(name).to_pylist()):
            raise ValueError(f'{name} chứa text null/rỗng; chưa tạo output.')
source_sha256 = hashlib.sha256(input_path.read_bytes()).hexdigest()
print(f'Input: {input_path.name} | {parquet.metadata.num_rows:,} dòng')
print('Các cột:', parquet.schema_arrow.names)
print('Thay text bằng vector trong:', TEXT_COLUMNS)


In [ ]:
def format_query(text):
    instruction = ENCODER['query_instruction']
    return f'Instruct: {instruction}\nQuery:{text}' if instruction else text


def forward_tokens(items):
    features = tokenizer.pad(items, padding=True, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        # Với left padding, vị trí cuối luôn là token thật cuối cùng.
        hidden = model(**features, use_cache=False).last_hidden_state
        vectors = F.normalize(hidden[:, -1].float(), p=2, dim=1)
    return vectors.cpu().numpy().astype(np.float16)


def forward_with_retry(items):
    try:
        return forward_tokens(items)
    except torch.cuda.OutOfMemoryError:
        if len(items) == 1:
            raise RuntimeError('GPU hết VRAM ở batch=1; không bỏ qua query.') from None
    # Ra khỏi except trước khi thu hồi tensor được giữ bởi exception traceback.
    gc.collect()
    torch.cuda.empty_cache()
    middle = len(items) // 2
    print(f'GPU OOM: chia batch {len(items)} → {middle} + {len(items)-middle}')
    return np.concatenate([forward_with_retry(items[:middle]), forward_with_retry(items[middle:])])


def encode_texts(texts):
    token_ids = tokenizer(
        [format_query(text) for text in texts], truncation=False,
        return_attention_mask=False, return_token_type_ids=False,
    )['input_ids']
    lengths = [len(ids) for ids in token_ids]
    if max(lengths) > ENCODER['max_length']:
        raise ValueError(f'Query có {max(lengths)} token; vượt max_length, không cắt text.')
    order = sorted(range(len(texts)), key=lengths.__getitem__, reverse=True)
    result = np.empty((len(texts), ENCODER['dimension']), dtype=np.float16)
    offset = 0
    while offset < len(order):
        size = min(ENCODER_BATCH_SIZE, max(1, PADDED_TOKEN_BUDGET // lengths[order[offset]]))
        positions = order[offset:offset + size]
        items = [{'input_ids': token_ids[index]} for index in positions]
        result[positions] = forward_with_retry(items)
        offset += len(positions)
    if not np.isfinite(result).all():
        raise ValueError('Embedding có NaN/Inf; chưa lưu output.')
    return result


def validate_output(path):
    result = pq.ParquetFile(path)
    if result.schema_arrow.names != parquet.schema_arrow.names:
        raise ValueError('Tên/thứ tự cột đã thay đổi.')
    if result.metadata.num_rows != parquet.metadata.num_rows:
        raise ValueError('Số dòng output không khớp input.')
    vector_type = pa.list_(pa.float16(), ENCODER['dimension'])
    for name in TEXT_COLUMNS:
        if result.schema_arrow.field(name).type != vector_type:
            raise ValueError(f'{name} không phải vector 1024 chiều float16.')
    untouched = [name for name in parquet.schema_arrow.names if name not in TEXT_COLUMNS]
    for name in untouched:
        if result.schema_arrow.field(name) != parquet.schema_arrow.field(name):
            raise ValueError(f'Kiểu dữ liệu/metadata cột {name} đã thay đổi.')
    source_batches = parquet.iter_batches(batch_size=ROW_BATCH_SIZE)
    output_batches = result.iter_batches(batch_size=ROW_BATCH_SIZE)
    for original, embedded in zip_longest(source_batches, output_batches):
        if original is None or embedded is None or original.num_rows != embedded.num_rows:
            raise ValueError('Số dòng trong batch không khớp.')
        for name in untouched:
            if not original.column(name).equals(embedded.column(name)):
                raise ValueError(f'Giá trị/thứ tự cột {name} đã thay đổi.')
        for name in TEXT_COLUMNS:
            column = embedded.column(name)
            values = np.asarray(column.to_pylist(), dtype=np.float32)
            if column.null_count or not np.isfinite(values).all():
                raise ValueError(f'{name}: vector null/NaN/Inf.')
            if not np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=0.002, rtol=0):
                raise ValueError(f'{name}: vector chưa chuẩn hóa L2.')
    return result.metadata.num_rows


In [ ]:
from transformers import AutoModel, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime → Change runtime type → T4 GPU rồi Run all.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')
tokenizer = AutoTokenizer.from_pretrained(
    ENCODER['model'], revision=ENCODER['revision'], padding_side='left',
)
model = AutoModel.from_pretrained(
    ENCODER['model'], revision=ENCODER['revision'],
    torch_dtype=torch.float16, attn_implementation='sdpa',
).eval().to('cuda')
model.config.use_cache = False
if model.config.hidden_size != ENCODER['dimension'] or next(model.parameters()).dtype != torch.float16:
    raise ValueError('Model không đúng 1024 chiều / FP16.')
print('Đã nạp Qwen3 Embedding 0.6B, FP16, SDPA, last-token pooling.')


In [ ]:
started = perf_counter()
# pandas metadata cũ mô tả text; bỏ phần mô tả lỗi thời sau khi thay kiểu cột.
metadata = dict(parquet.schema_arrow.metadata or {})
metadata.pop(b'pandas', None)
metadata[b'query_embedding'] = json.dumps({
    'encoder': ENCODER, 'source_sha256': source_sha256,
    'replaced_columns': list(TEXT_COLUMNS),
    'rows': parquet.metadata.num_rows, 'torch_version': torch.__version__,
}, ensure_ascii=False, sort_keys=True).encode('utf-8')
vector_type = pa.list_(pa.float16(), ENCODER['dimension'])
schema = pa.schema([
    pa.field(field.name, vector_type, nullable=field.nullable, metadata=field.metadata)
    if field.name in TEXT_COLUMNS else field
    for field in parquet.schema_arrow
], metadata=metadata)
output_path.parent.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(dir=output_path.parent, suffix='.parquet.partial', delete=False) as handle:
    temporary_path = Path(handle.name)
try:
    with pq.ParquetWriter(temporary_path, schema, compression='zstd') as writer:
        with tqdm(total=parquet.metadata.num_rows, desc='Embedding query', unit='dòng') as progress:
            for batch in parquet.iter_batches(batch_size=ROW_BATCH_SIZE):
                table = pa.Table.from_batches([batch])
                for name in TEXT_COLUMNS:
                    vectors = encode_texts(batch.column(name).to_pylist())
                    values = pa.array(vectors.reshape(-1), type=pa.float16())
                    arrays = pa.FixedSizeListArray.from_arrays(values, ENCODER['dimension'])
                    index = table.schema.get_field_index(name)
                    table = table.set_column(index, schema.field(name), arrays)
                writer.write_table(table.replace_schema_metadata(metadata))
                progress.update(batch.num_rows)
    count = validate_output(temporary_path)
    temporary_path.replace(output_path)
finally:
    temporary_path.unlink(missing_ok=True)
print(f'PASS: {count:,} dòng; {count*len(TEXT_COLUMNS):,} vector × 1024; Parquet float16; L2.')
print('Giữ nguyên cột, id, dense_weight và thứ tự dòng.')
print(f'Output: {output_path} | {output_path.stat().st_size / 1024**2:.2f} MiB')
print(f'Thời gian: {perf_counter()-started:.1f} giây')
print(pq.ParquetFile(output_path).schema_arrow)


In [ ]:
if DOWNLOAD_OUTPUT:
    import base64
    from IPython.display import HTML, display
    encoded = base64.b64encode(output_path.read_bytes()).decode('ascii')
    display(HTML(f'<a download="{output_path.name}" href="data:application/octet-stream;base64,{encoded}">Tải file Parquet FP16</a>'))
    del encoded


## Đọc file kết quả

```python
import json
import numpy as np
import pyarrow.parquet as pq

table = pq.read_table('query_rewritten_weighted_qwen3_1024_fp16.parquet')
info = json.loads(table.schema.metadata[b'query_embedding'])
# Đọc theo batch nếu file lớn; vector trong Parquet là fixed_size_list<halffloat>[1024].
row = table.slice(0, 1).to_pylist()[0]
vector_vi = np.asarray(row['query'], dtype=np.float16)
vector_en = np.asarray(row['english'], dtype=np.float16)
vector_zh = np.asarray(row['chinese'], dtype=np.float16)
```

FP16 có sai số làm tròn nhỏ; notebook kiểm tra chuẩn L2 với sai số 0,002.
`dense_weight` được giữ nguyên làm trọng số cho bước truy xuất sau.
Nếu runtime bị ngắt trước khi tải output, chạy lại; file đầu vào trên máy vẫn nguyên vẹn.
